# Run the HydroCNHS FE Walter zero-shot example / 執行 FE Walter zero-shot

## Goal / 目標
Repository YAML → hydrocnhs.Model.run → Dam observation → preserved prompt/schema → requested release → shared physics → downstream routing → audit/trace. The adapter supplies decision logic; HydroCNHS runs the simulation using frozen inflow. Offline replies are synthetic, not model outputs. The actual prompt is prompts/fe_walter_reference/zeroshot_system.txt.

Repository YAML → hydrocnhs.Model.run → Dam observation → 保留的 prompt／schema → 請求放流 → 共同物理 → 下游 routing → audit／trace。Adapter 提供決策邏輯，HydroCNHS 使用固定入流模擬。離線回覆是合成資料；system prompt 位於 prompts/fe_walter_reference/zeroshot_system.txt。

## Setup / 準備
Python 3.12 with repository dependencies. Local files only; no provider calls. / 使用 Python 3.12 與既有依賴；使用局部檔案，不呼叫 provider。

In [1]:
from pathlib import Path
import json, sys
ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p/'src/hydrocnhs').is_dir())
sys.path.insert(0, str(ROOT/'src'))
sys.path.insert(0, str(ROOT))
from llm_reservoir_agent.workflow import load_settings, run_settings, verify_settings_output
from llm_reservoir_agent.verification import verify_run
from unittest.mock import patch
from tutorials.inspect_decision_flow import show_decision_flow
def ensure_offline(config):
    settings = load_settings(ROOT/config)
    assert settings.report['planned_llm_calls'] == 0
    output = Path(settings.report['output'])
    if not output.exists():
        with patch('urllib.request.urlopen', side_effect=AssertionError('network forbidden')):
            run_settings(settings)
    return settings, verify_settings_output(settings)
print('Checkout:', ROOT.name)


Checkout: HydroCNHS-Delaware-Operations


## Steps

### Inspect input, system prompt and response fixture / 查看輸入、system prompt 與回覆 fixture

In [2]:
print((ROOT/'examples/fe_walter/zero_shot/experiment.hydroopslm.offline.yaml').read_text())
print((ROOT/'prompts/fe_walter_reference/zeroshot_system.txt').read_text())
fixture=json.loads((ROOT/'tests/fixtures/zeroshot_responses.json').read_text())
print('Reply kind:', fixture['data_kind']); print('First reply:', fixture['responses'][0])

# HydroOpsLM offline teaching run; original settings/results are preserved.
# FE Walter V3.7 zeroshot: original prompt builders, synthetic replies, zero model calls.
schema_version: llm-reservoir-experiment.v1
case: ../../../cases/fe_walter_reference/case.json # Frozen hydrology sample; V3.7 physics.
run_id: hydroopslm-fe-walter-zero-shot        # Change with output before a new run.
output: ../../../runs/hydroopslm-fe-walter-zero-shot
max_calls: 0                             # This example cannot call a provider.
auditors:
  release_limits:
    type: release_bounds
    version: "1"
agents:
  fe_operator:
    reservoir: FEWalter                  # Reservoir ID in case.json.
    mode: offline                        # Native prompt/audit run on fixture replies.
    strategy: reference_v3_7             # Generic strategy selector; optional vendor plugin.
    reference:
      profile: fe_walter                 # This reference profile is FE-specific.
      method: zeroshot                 

### Run daily and inspect rendered trace / Daily 執行與 prompt trace

In [3]:
settings, verified=ensure_offline('examples/fe_walter/zero_shot/experiment.hydroopslm.offline.yaml')
print(json.dumps(verified,indent=2))
raw=[json.loads(p.read_text()) for p in (Path(settings.report['output'])/'raw').glob('policy_*.json')]
first=min(raw,key=lambda item:item['backend_trace']['input_variables']['timestep'])
print('Daily variables:',first['backend_trace']['input_variables'])
print('Schema fields:', list(first['backend_trace']['schema']['properties']))

{
  "status": "PASS",
  "days": 3,
  "reservoir_count": 1,
  "reservoir_days": 3,
  "date_start": "2016-01-01",
  "date_end": "2016-01-03",
  "audit_counts": {
    "PASS": 3
  },
  "max_reservoir_balance_residual_m3": 0.0,
  "max_system_balance_residual_m3": 0.0,
  "max_routing_difference_cms": 0.0,
  "final_channel_storage_m3": 0.0,
  "observed_network_calls": 0,
  "operating_strategy_accepted": null,
  "audit_binding_counts": {
    "versioned_suite": 3,
    "legacy_unbound": 0
  },
  "external_audit_pins_checked": []
}
Daily variables: {'date': '2016-01-01', 'history_semantics': 'prior requested and implemented outflows; appended after physical commit', 'inflow_cfs': 1134.4949071013637, 'inflow_cms': 32.12531870666853, 'native_history': [], 'pool_elevation_start_ft': 1300.0, 'reservoir_profile': 'fe_walter', 'schema_version': 'reservoir-daily-inputs.v1', 'storage_start_af': 1992.61, 'storage_start_m3': 2457848.279729185, 'timestep': 0}
Schema fields: ['applicable_target_ft', 'intende

### Follow info, prompt, proposal, checker, audit/trace and feedback / 追查 info、prompt、proposal、checker、audit／trace 與 feedback

In [4]:
flow=show_decision_flow(Path(settings.report['output']))
assert flow['checker']['action_contract_recheck']=='PASS'
assert flow['next_info']['state']['storage_m3']==flow['execution']['storage_end_m3']
print('Original audit preserved:', flow['audit']['status'])


=== decision_id ===
a732c04171711b5950f2232b54b194b6056fff9014ae4d21632308c892f068d0

=== data_kind ===
synthetic_fixture

=== info ===
{
  "context": {
    "evaporation_m3": 0.0,
    "reservoir_id": "fe_walter",
    "seepage_m3": 0.0
  },
  "inflow_cms": 32.12531870666853,
  "release_upper_bound_cms": 60.572065702675154,
  "schema_version": "reservoir-observation.v1",
  "state": {
    "capacity_m3": 135532852.33895192,
    "dead_storage_m3": 49.3392742128,
    "dt_seconds": 86400.0,
    "maximum_release_cms": 283.16847,
    "minimum_release_cms": 1.41584235,
    "storage_m3": 2457848.279729185
  },
  "timestamp": "2016-01-01T00:00:00",
  "timestep": 0
}

=== prompt_variables ===
{
  "date": "2016-01-01",
  "history_semantics": "prior requested and implemented outflows; appended after physical commit",
  "inflow_cfs": 1134.4949071013637,
  "inflow_cms": 32.12531870666853,
  "native_history": [],
  "pool_elevation_start_ft": 1300.0,
  "reservoir_profile": "fe_walter",
  "schema_version

### Reject an invalid proposal before execution / 在執行前拒絕錯誤 proposal

In [5]:
from llm_reservoir_agent.contracts import parse_action
try:
    parse_action({'schema_version':'reservoir-action.v1','release_cms':-1})
except ValueError as error:
    print('Checker rejected negative release:', str(error))
else:
    raise AssertionError('invalid action accepted')
print('This isolated contract check calls no solver and changes no saved state.')

Checker rejected negative release: release_cms must be non-negative
This isolated contract check calls no solver and changes no saved state.


## Checks / 檢查
Review printed inputs, counts, sources and residuals. Synthetic replies validate interfaces only. / 核對輸入、計數、來源與殘差；合成回覆僅驗證介面。

## Next steps / 下一步
A new reservoir may use an explicit reservoir profile; it must supply its own facts and level-storage relation. Never copy FE Walter capacities or target levels to it.

其他水庫須提供自己的 profile、公開資料及水位／蓄水關係；不要複製 FE Walter 庫容或目標水位。